# 🛡️ DeepShield Audio — Notebook 01: Exploratory Data Analysis

Analyse the ASVspoof 2019 LA dataset:
- Dataset size and class distribution
- Spoofing algorithm breakdown (A01–A19)
- Speaker distribution
- Sample waveform + spectrogram per class
- Duration distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path
from tqdm import tqdm

from src.data_parser import load_all_splits, get_class_distribution, get_system_distribution, verify_no_leakage
from src.config import SAMPLE_RATE, MAX_AUDIO_SECONDS
from src.utils import setup_logging, plot_waveform, plot_spectrogram
from src.preprocessor import load_audio, extract_log_mel, pad_or_trim

setup_logging('INFO')
sns.set_style('darkgrid')
plt.rcParams['figure.facecolor'] = '#1E1E2E'
plt.rcParams['axes.facecolor'] = '#1E1E2E'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print('✅ Imports successful')

## 1. Load All Splits

In [ ]:
try:
    splits = load_all_splits()
    print('Dataset loaded from official protocol files.')
except FileNotFoundError as e:
    print(f'⚠️  Dataset not found: {e}')
    print('Creating synthetic demo data...')
    from src.data_parser import make_synthetic_dataframe
    splits = {
        'train': make_synthetic_dataframe(n_bonafide=2580, n_spoof=22800),
        'dev':   make_synthetic_dataframe(n_bonafide=2548, n_spoof=22296),
        'eval':  make_synthetic_dataframe(n_bonafide=7355, n_spoof=63882),
    }

train_df, dev_df, eval_df = splits['train'], splits['dev'], splits['eval']
all_df = pd.concat(splits.values(), ignore_index=True)

print(f'\nDataset Summary:')
print(f'  Train: {len(train_df):>7,} samples')
print(f'  Dev:   {len(dev_df):>7,} samples')
print(f'  Eval:  {len(eval_df):>7,} samples')
print(f'  TOTAL: {len(all_df):>7,} samples')

## 2. Class Distribution per Split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#2A9D8F', '#E63946']

for ax, (split_name, df) in zip(axes, splits.items()):
    dist = get_class_distribution(df)
    bars = ax.bar(dist.index, dist.values, color=colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, dist.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
    ax.set_title(f'{split_name.upper()} Split', fontsize=13, fontweight='bold', color='#4ECDC4')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ratio = dist.get('spoof', 0) / max(dist.get('bonafide', 1), 1)
    ax.text(0.5, 0.95, f'Spoof/Bonafide ratio: {ratio:.1f}x',
            transform=ax.transAxes, ha='center', va='top',
            color='#E9C46A', fontsize=9)

plt.suptitle('ASVspoof 2019 LA — Class Distribution per Split', fontsize=15, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#1E1E2E')
plt.show()

print('Class distribution saved.')

## 3. Spoofing Algorithm Breakdown

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (split_name, df) in zip(axes, splits.items()):
    sys_dist = get_system_distribution(df)
    if len(sys_dist) == 0:
        ax.text(0.5, 0.5, 'No spoof samples', ha='center', va='center', color='white')
        continue
    bars = ax.barh(sys_dist.index, sys_dist.values, color='#E63946', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{split_name.upper()} — Spoofing Systems', fontsize=12, fontweight='bold', color='#4ECDC4')
    ax.set_xlabel('Count')
    ax.set_ylabel('System ID')

plt.suptitle('Spoofing Algorithm Distribution (A01–A19)', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig('../results/system_distribution.png', dpi=150, bbox_inches='tight', facecolor='#1E1E2E')
plt.show()

## 4. Data Leakage Verification

In [ ]:
try:
    result = verify_no_leakage(splits)
    print('✅ VERIFIED: No data leakage between train/dev/eval splits.')
except AssertionError as e:
    print(f'❌ DATA LEAKAGE DETECTED: {e}')
except Exception as e:
    print(f'Leakage check skipped (synthetic data): {e}')

## 5. Sample Waveform + Spectrogram Visualisation

In [ ]:
# Visualise samples if files exist on disk
import os

bonafide_samples = train_df[train_df['label_str'] == 'bonafide']
spoof_samples    = train_df[train_df['label_str'] == 'spoof']

# Find files that actually exist
def find_existing(df, n=2):
    existing = []
    for _, row in df.iterrows():
        if os.path.exists(row['file_path']):
            existing.append(row)
        if len(existing) >= n:
            break
    return existing

real_files = find_existing(bonafide_samples)
fake_files = find_existing(spoof_samples)

if real_files or fake_files:
    n_total = len(real_files) + len(fake_files)
    fig, axes = plt.subplots(n_total, 2, figsize=(14, n_total * 3))
    if n_total == 1:
        axes = axes[np.newaxis, :]

    row = 0
    for samples, label_str in [(real_files, 'REAL'), (fake_files, 'FAKE')]:
        for sample in samples:
            try:
                wav = load_audio(sample['file_path'])
                wav_trimmed = pad_or_trim(wav)
                spec = extract_log_mel(wav_trimmed)

                # Waveform
                plot_waveform(wav, sr=SAMPLE_RATE,
                              title=f'{label_str} — Waveform ({sample["file_id"]})',
                              ax=axes[row, 0])
                # Spectrogram
                plot_spectrogram(spec, title=f'{label_str} — Log-Mel Spectrogram', ax=axes[row, 1])
                row += 1
            except Exception as e:
                print(f'Error loading {sample["file_path"]}: {e}')

    plt.tight_layout()
    plt.savefig('../results/sample_visualisations.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No audio files found on disk. Download the dataset to visualise samples.')
    print('The synthetic DataFrame does not contain real audio files.')

## 6. EDA Summary

In [ ]:
print('=' * 60)
print('  DEEPSHIELD AUDIO — EDA SUMMARY')
print('=' * 60)
for split_name, df in splits.items():
    dist = get_class_distribution(df)
    n_spoof = dist.get('spoof', 0)
    n_bon   = dist.get('bonafide', 0)
    print(f'  {split_name.upper():5s}: {len(df):>7,} total | '
          f'bonafide={n_bon:>6,} | spoof={n_spoof:>6,} | '
          f'ratio={n_spoof/max(n_bon,1):.1f}x')
print('=' * 60)
print(f'  IMPORTANT: Highly imbalanced dataset!')
print(f'  → Use class_weights in training to compensate.')
print(f'  → EER is a better metric than accuracy.')
print('=' * 60)